# データ取り込み

In [1]:
import pandas as pd

import pandas as pd
# df(original_dfに名称変更）をpickleで読み込む
original_df = pd.read_pickle(r"./OCR_Processed_DF/20241222_all_ocr_processed_df.pkl")
display(original_df.head())

,filename,title,content,tableinfo,graphinfo
0,1_08-39_1章_元.pdf,"[{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 3, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 46, 'file_name': '1_08-39_1章_元.pdf', '...","[{'id': 1, 'file_name': '1_08-39_1章_元.pdf', 'f..."
1,1年_中学校理科_指導書_板書例_p1.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...",[]
2,1年_中学校理科_指導書_板書例_p10.pdf,"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 4, 'file_name': '1年_中学校理科_指導書_板書例_p10....",[]
3,1年_中学校理科_指導書_板書例_p100.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p100...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p100...",[],"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p100..."
4,1年_中学校理科_指導書_板書例_p101.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p101...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p101...",[],"[{'id': 23, 'file_name': '1年_中学校理科_指導書_板書例_p10..."


In [3]:
# original_df = spark.table("poc2.mateq_orijinal_data_0911").toPandas() # <- 読み込むデータ名を記入してください。
# original_df = spark.table("poc2.mateq_orijinal_data").toPandas() # <- 読み込むデータ名を記入してください。
# display(original_df)

print(original_df.dtypes)


filename     object
title        object
content      object
tableinfo    object
graphinfo    object
dtype: object


### title列が空の行の確認＆削除

In [4]:
def count_and_delete_empty_title(df):
    # title 列が空の行を数える
    empty_title_count = df['title'].isna().sum() + (df['title'] == '').sum()

    # title 列が空の行を削除
    no_empty_df = df[df['title'].notna() & (df['title'] != '')]
    print(f"Number of empty 'title' entries: {empty_title_count}")
    return no_empty_df

no_empty_df = count_and_delete_empty_title(original_df)
# display(no_empty_df)
print(f"length of original_df: {len(original_df)}")
print(f"length of no_empty_df: {len(no_empty_df)}")

Number of empty 'title' entries: 0
length of original_df: 445
length of no_empty_df: 445


# ケバとり

In [7]:
import unicodedata
import ast

def clean_text(text) -> str:
    """google style docstring
    Cleans the given text by removing unnecessary characters and spaces.
    Args:
        text (str): The text to clean.
    Returns:
        str: The cleaned text."""
    if not isinstance(text, str):
        return text
    # 不要な文字列のリスト
    garbage_list = ['　', ' ',' ']
    # 各不要な文字列を除去
    for garbage in garbage_list:
        text = text.replace(garbage, '')
    # 両端の空白を除去
    return text.strip()

def unicode_normalize(text: str) -> str:
        """
        Normalizes unicode characters in the given text.

        Parameters:
        - text (str): The text to normalize.

        Returns:
        - str: The normalized text.
        """
        if not isinstance(text, str):
            return text
        return unicodedata.normalize("NFKC", text)

def convert_dictlist_to_stringlist(dict_list) -> list:
    """google style docstring
    Converts a list of dictionaries to a list of strings.
    Args:
        dict_list (list): A list of dictionaries.
    Returns:
        list: A list of strings."""
    string_list = []
    for d in dict_list:
        string_list.append(f"{str(d)}")
    return string_list

def apply_cleaning_functions(df) -> pd.DataFrame:
    """google style docstring
    Applies the cleaning functions to the given DataFrame.
    Args:
        df (pd.DataFrame): The DataFrame to clean.
    Returns:
        pd.DataFrame: The cleaned DataFrame."""
    special_columns = ['content', 'tableinfo', 'graphinfo', 'title']
    updated_rows = []

    for row in df.itertuples(index=True, name='Pandas'):
        index = row.Index
        new_row = list(row)
        # majorcategory = getattr(row, 'majorcategory')

        for column in df.columns:
            col_index = df.columns.get_loc(column)
            if column in special_columns:
                cell_val = getattr(row, column)
                # Dictionaryのリストを文字列から抽出する
                list_of_dicts = [ast.literal_eval(item) for item in cell_val]
                # Dictionaryのリストをループし、各エントリーに clean_textとunicode_normalizeを実行する
                for dict in list_of_dicts:
                    # 'file_name' キーがdictionary にあることを確認
                    if 'file_name' in dict:
                        dict['file_name'] = clean_text(dict['file_name'])
                        dict['file_name'] = unicode_normalize(dict['file_name'])

                    # 'content' キーがdictionary にあることを確認
                    if 'content' in dict:
                        dict['content'] = clean_text(dict['content'])
                        dict['content'] = unicode_normalize(dict['content'])
                # Convert back to list of strings
                converted = convert_dictlist_to_stringlist(list_of_dicts)
                new_row[col_index + 1] = converted  # +1 because itertuples includes the index as the first element
            else:
                new_row[col_index + 1] = unicode_normalize(clean_text(getattr(row, column)))

        updated_rows.append(new_row)

    # Create a new DataFrame from the updated rows
    updated_df = pd.DataFrame(updated_rows, columns=['Index'] + list(df.columns)).set_index('Index')
    return updated_df

# test_df = no_empty_df.loc[900:930]
# kebatori_df = apply_cleaning_functions(test_df)
kebatori_df = apply_cleaning_functions(no_empty_df)

# Assert that the DataFrames are identical
assert not original_df.equals(kebatori_df), "orignal_df と kebatori_dfが同じようです。ケバとりが実施されていない可能性があります。"

display(no_empty_df)
display(kebatori_df.loc[:])
# display(original_df)

,filename,title,content,tableinfo,graphinfo
0,1_08-39_1章_元.pdf,"[{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 3, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 46, 'file_name': '1_08-39_1章_元.pdf', '...","[{'id': 1, 'file_name': '1_08-39_1章_元.pdf', 'f..."
1,1年_中学校理科_指導書_板書例_p1.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...",[]
2,1年_中学校理科_指導書_板書例_p10.pdf,"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 4, 'file_name': '1年_中学校理科_指導書_板書例_p10....",[]
3,1年_中学校理科_指導書_板書例_p100.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p100...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p100...",[],"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p100..."
4,1年_中学校理科_指導書_板書例_p101.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p101...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p101...",[],"[{'id': 23, 'file_name': '1年_中学校理科_指導書_板書例_p10..."
...,...,...,...,...,...
440,3年_中学校理科_指導書_板書例_p95.pdf,"[{'id': 3, 'file_name': '3年_中学校理科_指導書_板書例_p95....","[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p95....",[],"[{'id': 2, 'file_name': '3年_中学校理科_指導書_板書例_p95...."
441,3年_中学校理科_指導書_板書例_p96.pdf,"[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p96....","[{'id': 2, 'file_name': '3年_中学校理科_指導書_板書例_p96....",[],"[{'id': 4, 'file_name': '3年_中学校理科_指導書_板書例_p96...."
442,3年_中学校理科_指導書_板書例_p97.pdf,"[{'id': 4, 'file_name': '3年_中学校理科_指導書_板書例_p97....","[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p97....",[],"[{'id': 3, 'file_name': '3年_中学校理科_指導書_板書例_p97...."
443,3年_中学校理科_指導書_板書例_p98.pdf,"[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p98....","[{'id': 2, 'file_name': '3年_中学校理科_指導書_板書例_p98....",[],"[{'id': 4, 'file_name': '3年_中学校理科_指導書_板書例_p98...."


,filename,title,content,tableinfo,graphinfo
Index,,,,,
0,1_08-39_1章_元.pdf,"[{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 3, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 46, 'file_name': '1_08-39_1章_元.pdf', '...","[{'id': 1, 'file_name': '1_08-39_1章_元.pdf', 'f..."
1,1年_中学校理科_指導書_板書例_p1.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p1.p...",[]
2,1年_中学校理科_指導書_板書例_p10.pdf,"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p10....","[{'id': 4, 'file_name': '1年_中学校理科_指導書_板書例_p10....",[]
3,1年_中学校理科_指導書_板書例_p100.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p100...","[{'id': 3, 'file_name': '1年_中学校理科_指導書_板書例_p100...",[],"[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p100..."
4,1年_中学校理科_指導書_板書例_p101.pdf,"[{'id': 1, 'file_name': '1年_中学校理科_指導書_板書例_p101...","[{'id': 2, 'file_name': '1年_中学校理科_指導書_板書例_p101...",[],"[{'id': 23, 'file_name': '1年_中学校理科_指導書_板書例_p10..."
...,...,...,...,...,...
440,3年_中学校理科_指導書_板書例_p95.pdf,"[{'id': 3, 'file_name': '3年_中学校理科_指導書_板書例_p95....","[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p95....",[],"[{'id': 2, 'file_name': '3年_中学校理科_指導書_板書例_p95...."
441,3年_中学校理科_指導書_板書例_p96.pdf,"[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p96....","[{'id': 2, 'file_name': '3年_中学校理科_指導書_板書例_p96....",[],"[{'id': 4, 'file_name': '3年_中学校理科_指導書_板書例_p96...."
442,3年_中学校理科_指導書_板書例_p97.pdf,"[{'id': 4, 'file_name': '3年_中学校理科_指導書_板書例_p97....","[{'id': 1, 'file_name': '3年_中学校理科_指導書_板書例_p97....",[],"[{'id': 3, 'file_name': '3年_中学校理科_指導書_板書例_p97...."


In [8]:
special_columns = ['content', 'tableinfo', 'graphinfo', 'title']

In [0]:
# test_df = no_empty_df.loc[900:930]
# display(test_df)

In [0]:
# sample_data = [
#     "{'content': '{\\n\"材料名\":\"シリンダ: 球状黒鉛鋳鉄（FCD700）、ピストン: クロムモリブデン鋼（SCM415）\",\\n\"題名\":\"材料リマインダ\",\\n\"No\":\"T072\",\\n\"表題\":\"油圧ポンプのシリンダとピストン間の油膜切れによる焼付き\",\\n\"設計名\":\"タンデム用油圧ポンプ\",\\n\"故障モード\":\"焼付き\",\\n\"部品名\":\"タンデム用油圧ポンプ\",\\n\"材料分類\":\"鋳鉄材、鋼鉄、その他鋼材\",\\n\"材料名\":\"シリンダ:球状黒鉛鋳鉄(FCD700)、ピストン:クロムモリブデン鋼(SCM415)\",\\n\"内容\":\"ダンプ油圧ポンプ(コンプレッサタイプ)において、シリンダーとピストン間の摺動にて焼付き発生。\\\\n・摺動状態低動\\\\n・推進体油圧オイル\",\\n\"要因\":\"油膜切れを起こし、同種材料の摺動により焼き付いた。\",\\n\"対策\":\"・シリンダ全面にリン酸マンガン処理を追加。\\\\n・シリンダ材質を、アルミの中での耐摩耗性の優れる高SiのA390-T6に変更。\",\\n\"備考\":\"同種材料の摺動は避ける。\"\\n}', 'figure_table_type': 'None', '材料リマインダ（Ｔ０７２）.pdf': 'None', 'file_path': '/dbfs/mnt/rag-poc-q2-container/pdfs/リマインダ_機密事項あり取り扱い注意/その他/材料リマインダ（Ｔ０７２）.pdf', 'id': 1, 'page_number': 1, 'region': 'None', 'type': 'その他の文章'}",
#     "{'content': '{\\n\"random\":\"（１８９９３　）あｂｆ691\",\\n\"題名\":\"材料リマインダ\",\\n\"No\":\"T068\",\\n\"表題\":\"すべり軸受の低硬さによるフレッチング摩耗\",\\n\"設計名\":\"故障モード摩耗\",\\n\"故障モード\":\"摩耗\",\\n\"部品名\":\"すべり軸受け\",\\n\"材料分類\":\"軟鋼材、銅合金\",\\n\"材料名\":\"銅合金\",\\n\"内容\":\"軸受が摩耗し、作動性悪化に至った。\",\\n\"要因\":\"摩耗粉の状態とCAE解析結果より、フレッチングによる摩耗と判断。\",\\n\"対策\":\"耐摩耗性向上のため材料高硬度化。\",\\n\"備考\":\"当該軸受部は常時摺動ではなく、ON-OFFの間欠作動。高硬度化による有弁反にも留意が必要。\"\\n}', 'figure_table_type': 'None', '材料リマインダ（Ｔ０６８）.pdf': 'None', 'file_path': '/dbfs/mnt/rag-poc-q2-container/pdfs/リマインダ_機密事項あり取り扱い注意/その他/材料リマインダ（Ｔ０６８）.pdf', 'id': 1, 'page_number': 1, 'region': 'None', 'type': 'その他の文章'}"
# ]


# list_of_dicts = [ast.literal_eval(item) for item in sample_data]
# for d in list_of_dicts:
#     print(f"{d}\n\n")
#     if 'content' in d:
#         cleaned = clean_text(d['content'])
#         d['content'] = cleaned
#         normalized = unicode_normalize(d['content'])
#         d['content'] = normalized
#         print(normalized)
    
# # print(list_of_dicts)
# print(list_of_dicts[0])
# print("\n")
# print(list_of_dicts[1])
# print("\n\n")
# print(list_of_dicts[0]['content'])
# print("\n\n")
# print(type(list_of_dicts[0]['content']))

In [11]:
# データフレームに保存
# kebatori_df.to_pickle(r"./Kebatori_DF/20241222_all_kebatori_df.pkl")


# データベースに保存

In [9]:
# from datetime import datetime, timedelta

# def save_df_to_catalog(df, table_name: str):
#     # 現在のUTC日時を取得
#     now_utc = datetime.utcnow()
#     # 日本時間 (UTC+9) に変換
#     now_japan = now_utc + timedelta(hours=9)
#     # yyyyMMddHHmm形式の文字列に変換
#     formatted_date_japan = now_japan.strftime('%Y%m%d_%H%M')

#     df = spark.createDataFrame(df)
#     # # Convert Pandas DataFrame to Spark DataFrame if necessary
#     # if isinstance(df, pd.DataFrame):
#     #     df = spark.createDataFrame(df)

#     # Create the schema in the specified catalog and schema
#     spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.poc2")

#     # Save the DataFrame as a table in the specified schema
#     df.write.mode("overwrite").saveAsTable(f"poc2.{table_name}_{formatted_date_japan}")
    
#     print(f"Table 'poc2.{table_name}_{formatted_date_japan}' was created.")
#     check_df = spark.table(f"poc2.{table_name}_{formatted_date_japan}")
#     display(check_df)

# save_df_to_catalog(kebatori_df, "kebatori_complete_data")

In [0]:
# from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# from datetime import datetime, timedelta

# # 現在のUTC日時を取得
# now_utc = datetime.utcnow()

# # 日本時間 (UTC+9) に変換
# now_japan = now_utc + timedelta(hours=9)

# # yyyyMMddHHmm形式の文字列に変換
# formatted_date_japan = now_japan.strftime('%Y%m%d_%H%M')

# display(formatted_date_japan)

# # Initialize Spark session
# spark = SparkSession.builder \
#     .appName("Create Schema") \
#     .enableHiveSupport() \
#     .getOrCreate()

# # Define the schema
# schema = StructType([
#     StructField("tite", StringType(), True),
#     StructField("majorcategory", StringType(), True),
#     StructField("subcategory", StringType(), True),
#     StructField("content", ArrayType(StringType()), True),
#     StructField("tableinfo", ArrayType(StringType()), True),
#     StructField("graphinfo", ArrayType(StringType()), True)
# ])


# # Create the schema in the specified catalog and schema
# spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.poc2")

# # Save the DataFrame as a table in the specified schema
# spark.createDataFrame(kebatori_df).write.mode("overwrite").saveAsTable(f"poc2.kebatori_complete_data_{formatted_date_japan}")